<a href="https://colab.research.google.com/github/stepthom/869_course/blob/main/ensemble/slides_ensemble_catboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Playground for Catboost

- Stephen W. Thomas
- Used for MMA 869, MMAI 869, and GMMA 869

This code is going to show a simple Catboost model and some of its features.

- Loading and splitting data
- Basic feature engineering
- CatBoost's native handling of categoricals and missing values
- Cross-validation for F1 estimation
- Holdout evaluation
- Early stopping


In [1]:
pip install catboost

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostClassifier

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/stepthom/869_course/main/data/bank.csv")
print(f"Dataset shape: {df.shape}")
print(f"\nColumn types:\n{df.dtypes}")
print(f"\nTarget distribution:\n{df['y'].value_counts()}")

Dataset shape: (4521, 17)

Column types:
age           int64
job          object
marital      object
education    object
default      object
balance       int64
housing      object
loan         object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object

Target distribution:
y
no     4000
yes     521
Name: count, dtype: int64


In [4]:
le = LabelEncoder()
df['y'] = le.fit_transform(df['y'])
print(f"Target encoding: {dict(zip(le.classes_, range(len(le.classes_))))}")

Target encoding: {'no': 0, 'yes': 1}


In [5]:
print("\n" + "=" * 60)
print("Splitting into train and holdout sets...")
print("=" * 60)

X = df.drop(columns=['y'])
y = df['y']

X_train, X_holdout, y_train, y_holdout = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Holdout set:  {X_holdout.shape[0]} samples")


Splitting into train and holdout sets...
Training set: 3616 samples
Holdout set:  905 samples


In [6]:
y

,y
0,0
1,0
2,0
3,0
4,0
...,...
4516,0
4517,0
4518,0
4519,0


In [7]:
# Identify Categorical Features for CatBoost
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"\nCategorical features ({len(cat_features)}): {cat_features}")


Categorical features (9): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


# Model without early stopping

## CV estimate

In [8]:
model1 = CatBoostClassifier(
    iterations=1000,
    random_seed=42,
    verbose=0,
    cat_features=cat_features,
)

# Cross-validation using sklearn
cv_scores = cross_val_score(
    model1,
    X_train,
    y_train,
    cv=5,
    scoring='f1_macro',
)

print(f"CV F1 Scores: {cv_scores.round(4)}")
print(f"CV F1 Mean:   {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

CV F1 Scores: [0.7516 0.7194 0.7478 0.7218 0.7095]
CV F1 Mean:   0.7300 (+/- 0.0333)


## Evaluation on Hold-Out

In [9]:
model1 = CatBoostClassifier(
    iterations=1000,
    random_seed=42,
    verbose=0,
    cat_features=cat_features,
)

model1.fit(X_train, y_train)

# Evaluate on holdout
y_pred_default = model1.predict(X_holdout)
f1_default = f1_score(y_holdout, y_pred_default, pos_label=1, average="macro")

print(f"\nHoldout F1 Score: {f1_default:.4f}")
print(f"\nClassification Report (Holdout):")
print(classification_report(y_holdout, y_pred_default))


Holdout F1 Score: 0.6933

Classification Report (Holdout):
              precision    recall  f1-score   support

           0       0.92      0.97      0.94       801
           1       0.59      0.36      0.44       104

    accuracy                           0.90       905
   macro avg       0.75      0.66      0.69       905
weighted avg       0.88      0.90      0.89       905



# Model with early stopping

## CV estimate

In [10]:
model2 = CatBoostClassifier(
    iterations=1000,
    early_stopping_rounds=50,
    random_seed=42,
    verbose=0,
    cat_features=cat_features,
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores_early = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model2.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

    y_pred = model2.predict(X_val)
    fold_f1 = f1_score(y_val, y_pred, pos_label=1, average="macro")
    cv_scores_early.append(fold_f1)

    print(f"Fold {fold}: F1={fold_f1:.4f}, Best iter={model2.get_best_iteration()}")

print(f"\nCV F1 Mean: {np.mean(cv_scores_early):.4f} (+/- {np.std(cv_scores_early) * 2:.4f})")

Fold 1: F1=0.6916, Best iter=245
Fold 2: F1=0.7060, Best iter=149
Fold 3: F1=0.7602, Best iter=500
Fold 4: F1=0.7044, Best iter=199
Fold 5: F1=0.6978, Best iter=188

CV F1 Mean: 0.7120 (+/- 0.0492)


## Evaluation on Hold-Out

In [11]:
model2 = CatBoostClassifier(
    iterations=1000,
    random_seed=42,
    verbose=0,
    cat_features=cat_features,
    early_stopping_rounds=50,
)

# Split training into train/validation for early stopping
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

model2.fit(
    X_tr, y_tr,
    eval_set=(X_val, y_val),
    use_best_model=True
)

print(f"\nBest iteration: {model2.get_best_iteration()}")
print(f"Total iterations trained: {model2.tree_count_}")

# Evaluate on holdout
y_pred_early = model2.predict(X_holdout)
f1_early = f1_score(y_holdout, y_pred_early, pos_label=1, average="macro")

print(f"\nHoldout F1 Score (Early Stopping): {f1_early:.4f}")
print(f"\nClassification Report (Holdout):")
print(classification_report(y_holdout, y_pred_early))

Training subset:   2892 samples
Validation subset: 724 samples

Best iteration: 246
Total iterations trained: 247

Holdout F1 Score (Early Stopping): 0.7062

Classification Report (Holdout):
              precision    recall  f1-score   support

           0       0.92      0.97      0.94       801
           1       0.60      0.38      0.47       104

    accuracy                           0.90       905
   macro avg       0.76      0.68      0.71       905
weighted avg       0.89      0.90      0.89       905

